# ISIC 2024 strict_input export audit

Updated: 2026-05-14

이 노트북은 Nested CV strict input export 계약을 검토하는 audit report다. 시작 노트북 `isic2024_strict_input_iddx_full_dataset_20260514.ipynb`에서 고정한 `cv_test_fold == outer_test`, `cv_train`, `inner_train`, `inner_validation` 계약을 실제 export artifact에서 다시 확인한다.


## 1. Audit scope

이 audit은 다음을 확인한다.

- `isic2024_strict_model_input.csv`가 ordinary inference-time `strict_input` feature table인지 확인
- `isic2024_iddx_full_train_only_sidecar.csv`가 row-aligned train-only sidecar인지 확인
- nested split artifact가 있으면 outer/inner patient overlap과 fold distribution 확인
- nested split artifact가 없으면 `PENDING_EXPORT_CLI`로 표시

현재 기본 protocol은 outer 5-fold, inner 4-fold patient-level Triple Stratified Nested CV다.


In [ ]:
from pathlib import Path
import json

import pandas as pd
try:
    from IPython.display import Markdown, display
except ModuleNotFoundError:
    def Markdown(value):
        return value

    def display(value):
        print(value)


def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / 'src' / 'isic2024_multimodal').exists():
            return candidate
    raise RuntimeError('Could not locate repository root from current working directory')


def first_existing_path(candidates):
    return next((path for path in candidates if path.exists()), candidates[0])


REPO_ROOT = find_repo_root()
TRAIN_METADATA_PATH = first_existing_path(
    [
        REPO_ROOT / 'data/raw/isic-2024-challenge/train-metadata.csv',
        REPO_ROOT / 'data/raw/isic_2024_challenge/train-metadata.csv',
        REPO_ROOT / 'data/raw/train-metadata.csv',
    ]
)
STRICT_INPUT_PATH = REPO_ROOT / 'data/processed/isic2024_strict_model_input.csv'
IDDX_SIDECAR_PATH = REPO_ROOT / 'data/processed/isic2024_iddx_full_train_only_sidecar.csv'
NESTED_SPLIT_PATH = REPO_ROOT / 'data/splits/isic2024_official_train_nested_5x4_seed42.csv'
SUMMARY_PATH = REPO_ROOT / 'experiments/evidence/validation_protocol/isic2024_strict_input_export_summary_seed42.json'

paths = {
    'official_train_metadata': TRAIN_METADATA_PATH,
    'strict_input': STRICT_INPUT_PATH,
    'iddx_sidecar': IDDX_SIDECAR_PATH,
    'nested_split': NESTED_SPLIT_PATH,
    'summary': SUMMARY_PATH,
}

path_audit_df = pd.DataFrame(
    [{'artifact': name, 'path': path.as_posix(), 'exists': path.exists()} for name, path in paths.items()]
)
display(Markdown('**표 1. audit target artifact existence**'))
display(path_audit_df)


## 2. Load artifacts

이 셀은 산출물을 읽기만 한다. raw data, processed data, split file을 수정하지 않는다. nested split artifact가 없으면 먼저 export CLI를 실행해야 한다.


In [ ]:
def read_optional_csv(path):
    return pd.read_csv(path, low_memory=False) if path.exists() else None


official_train_pool = read_optional_csv(TRAIN_METADATA_PATH)
strict_df = read_optional_csv(STRICT_INPUT_PATH)
iddx_sidecar_df = read_optional_csv(IDDX_SIDECAR_PATH)
nested_split_df = read_optional_csv(NESTED_SPLIT_PATH)
summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8')) if SUMMARY_PATH.exists() else None

loaded_artifacts = {
    'official_train_pool': official_train_pool,
    'strict_input': strict_df,
    'iddx_sidecar': iddx_sidecar_df,
    'nested_split': nested_split_df,
}
artifact_shape_df = pd.DataFrame(
    [
        {
            'artifact': name,
            'status': 'loaded' if frame is not None else 'missing_or_pending',
            'rows': len(frame) if frame is not None else pd.NA,
            'columns': frame.shape[1] if frame is not None else pd.NA,
        }
        for name, frame in loaded_artifacts.items()
    ]
)
display(Markdown('**표 2. loaded artifact shape**'))
display(artifact_shape_df)


## 3. Input leakage audit

Strict input table은 inference-time ordinary metadata만 포함해야 한다. `iddx_full`, diagnosis/reference columns, provenance/context columns, schema-only constant column은 main strict input에서 제외되어야 한다.


In [ ]:
disallowed_main_columns = {
    'iddx_full', 'iddx_1', 'iddx_2', 'iddx_3', 'iddx_4', 'iddx_5',
    'mel_mitotic_index', 'mel_thick_mm', 'tbp_lv_dnn_lesion_confidence',
    'attribution', 'copyright_license', 'image_type',
}

if strict_df is None or iddx_sidecar_df is None:
    input_leakage_audit_df = pd.DataFrame(
        [
            {
                'check': 'strict_input_and_sidecar_available',
                'pass': False,
                'status': 'PENDING_EXPORT_CLI',
                'value': 'Run strict input export before executing this audit fully.',
            }
        ]
    )
else:
    input_leakage_audit_df = pd.DataFrame(
        [
            {'check': 'strict_input_has_isic_id_unique', 'pass': bool(strict_df['isic_id'].is_unique), 'value': strict_df['isic_id'].nunique()},
            {'check': 'sidecar_aligned_by_isic_id', 'pass': bool(strict_df['isic_id'].equals(iddx_sidecar_df['isic_id'])), 'value': len(iddx_sidecar_df)},
            {'check': 'iddx_full_excluded_from_strict_input', 'pass': 'iddx_full' not in strict_df.columns, 'value': 'iddx_full' in strict_df.columns},
            {'check': 'diagnosis_reference_columns_excluded', 'pass': not bool(set(strict_df.columns) & disallowed_main_columns), 'value': sorted(set(strict_df.columns) & disallowed_main_columns)},
            {'check': 'sidecar_has_iddx_full_train_only', 'pass': 'iddx_full_train_only' in iddx_sidecar_df.columns, 'value': 'iddx_full_train_only' in iddx_sidecar_df.columns},
        ]
    )

display(Markdown('**표 3. strict input / iddx sidecar leakage audit**'))
display(input_leakage_audit_df)
if strict_df is not None and iddx_sidecar_df is not None and not input_leakage_audit_df['pass'].all():
    raise AssertionError('Input leakage audit failed')


## 4. official_train_pool alignment audit

Nested CV protocol에서는 `train-metadata.csv` 전체가 `official_train_pool`이다. strict input과 sidecar는 이 pool과 row-aligned되어야 하며, 별도 고정 internal test split을 기본 계약으로 요구하지 않는다.


In [ ]:
if official_train_pool is None:
    pool_alignment_df = pd.DataFrame(
        [{'check': 'official_train_pool_available', 'pass': False, 'status': 'MISSING_RAW_METADATA', 'value': TRAIN_METADATA_PATH.as_posix()}]
    )
else:
    pool_alignment_rows = [
        {'check': 'official_train_pool_has_unique_isic_id', 'pass': bool(official_train_pool['isic_id'].is_unique), 'value': official_train_pool['isic_id'].nunique()},
        {'check': 'official_train_pool_has_patient_id', 'pass': bool(official_train_pool['patient_id'].notna().all()), 'value': int(official_train_pool['patient_id'].isna().sum())},
        {'check': 'official_train_pool_target_binary', 'pass': set(official_train_pool['target'].dropna().unique()).issubset({0, 1}), 'value': sorted(official_train_pool['target'].dropna().unique().tolist())},
    ]
    if strict_df is not None:
        pool_alignment_rows.append({'check': 'strict_input_row_count_matches_official_train_pool', 'pass': len(strict_df) == len(official_train_pool), 'value': {'strict_rows': len(strict_df), 'official_train_pool_rows': len(official_train_pool)}})
        pool_alignment_rows.append({'check': 'strict_input_isic_ids_match_official_train_pool', 'pass': set(strict_df['isic_id']) == set(official_train_pool['isic_id']), 'value': len(set(strict_df['isic_id']) ^ set(official_train_pool['isic_id']))})
    if iddx_sidecar_df is not None:
        pool_alignment_rows.append({'check': 'sidecar_isic_ids_match_official_train_pool', 'pass': set(iddx_sidecar_df['isic_id']) == set(official_train_pool['isic_id']), 'value': len(set(iddx_sidecar_df['isic_id']) ^ set(official_train_pool['isic_id']))})
    pool_alignment_df = pd.DataFrame(pool_alignment_rows)

display(Markdown('**표 4. official_train_pool alignment audit**'))
display(pool_alignment_df)
if official_train_pool is not None and not pool_alignment_df['pass'].all():
    raise AssertionError('official_train_pool alignment audit failed')


## 5. Nested CV split audit

`data/splits/isic2024_official_train_nested_5x4_seed42.csv`가 생성되면 이 셀이 outer/inner role 분포와 patient overlap을 확인한다. 파일이 없으면 먼저 export CLI를 실행한다.


In [ ]:
required_nested_columns = {'isic_id', 'patient_id', 'lesion_id', 'outer_fold', 'cv_test_fold', 'inner_fold', 'split_role'}
allowed_split_roles = {'outer_test', 'inner_train', 'inner_validation'}

if nested_split_df is None:
    nested_status_df = pd.DataFrame(
        [
            {
                'check': 'nested_split_artifact_available',
                'pass': False,
                'status': 'PENDING_EXPORT_CLI',
                'value': NESTED_SPLIT_PATH.as_posix(),
            }
        ]
    )
    display(Markdown('**표 5-a. Nested CV split implementation status**'))
    display(nested_status_df)
else:
    nested_split_df = nested_split_df.copy()
    missing_nested_columns = sorted(required_nested_columns - set(nested_split_df.columns))
    if not missing_nested_columns:
        nested_split_df['outer_fold'] = nested_split_df['outer_fold'].astype(int)
        nested_split_df['cv_test_fold'] = nested_split_df['cv_test_fold'].astype(int)
        nested_split_df['inner_fold'] = nested_split_df['inner_fold'].astype(int)
        nested_split_df['isic_id'] = nested_split_df['isic_id'].astype(str)
        nested_split_df['patient_id'] = nested_split_df['patient_id'].astype(str)

    role_values = set(nested_split_df['split_role'].astype(str)) if 'split_role' in nested_split_df.columns else set()
    duplicate_role_rows = (
        int(nested_split_df.duplicated(['isic_id', 'outer_fold', 'inner_fold']).sum())
        if not missing_nested_columns
        else pd.NA
    )
    nested_schema_audit_df = pd.DataFrame(
        [
            {'check': 'required_nested_split_columns_present', 'pass': not missing_nested_columns, 'value': missing_nested_columns},
            {'check': 'split_role_values_allowed', 'pass': role_values.issubset(allowed_split_roles), 'value': sorted(role_values)},
            {'check': 'cv_test_fold_equals_outer_fold', 'pass': bool((nested_split_df['cv_test_fold'] == nested_split_df['outer_fold']).all()) if not missing_nested_columns else False, 'value': 'cv_test_fold == outer_fold'},
            {'check': 'outer_fold_has_5_values', 'pass': nested_split_df['outer_fold'].nunique() == 5 if not missing_nested_columns else False, 'value': sorted(nested_split_df['outer_fold'].dropna().unique().tolist()) if 'outer_fold' in nested_split_df.columns else []},
            {'check': 'inner_fold_has_4_values', 'pass': nested_split_df['inner_fold'].nunique() == 4 if not missing_nested_columns else False, 'value': sorted(nested_split_df['inner_fold'].dropna().unique().tolist()) if 'inner_fold' in nested_split_df.columns else []},
            {'check': 'one_role_row_per_isic_outer_inner', 'pass': duplicate_role_rows == 0, 'value': duplicate_role_rows},
        ]
    )
    if official_train_pool is not None and not missing_nested_columns:
        expected_nested_rows = len(official_train_pool) * nested_split_df['outer_fold'].nunique() * nested_split_df['inner_fold'].nunique()
        nested_schema_audit_df = pd.concat(
            [
                nested_schema_audit_df,
                pd.DataFrame(
                    [
                        {'check': 'nested_split_long_row_count_matches_pool_x_outer_x_inner', 'pass': len(nested_split_df) == expected_nested_rows, 'value': {'nested_rows': len(nested_split_df), 'expected_rows': expected_nested_rows}},
                        {'check': 'nested_split_isic_ids_match_official_train_pool', 'pass': set(nested_split_df['isic_id']) == set(official_train_pool['isic_id'].astype(str)), 'value': len(set(nested_split_df['isic_id']) ^ set(official_train_pool['isic_id'].astype(str)))},
                    ]
                ),
            ],
            ignore_index=True,
        )

    nested_summary_source_df = nested_split_df
    if strict_df is not None and 'target' in strict_df.columns and not missing_nested_columns:
        nested_summary_source_df = nested_split_df.merge(
            strict_df[['isic_id', 'target']], on='isic_id', how='left', validate='many_to_one'
        )

    outer_source_df = nested_summary_source_df.assign(
        outer_role=nested_summary_source_df['split_role'].where(
            nested_summary_source_df['split_role'].eq('outer_test'),
            'cv_train',
        )
    ).drop_duplicates(['isic_id', 'outer_fold', 'outer_role'])
    outer_agg_spec = {'rows': ('isic_id', 'nunique'), 'patients': ('patient_id', 'nunique')}
    if 'target' in outer_source_df.columns:
        outer_agg_spec['positive_rows'] = ('target', 'sum')
    outer_summary_df = (
        outer_source_df.groupby(['outer_fold', 'outer_role'], dropna=False)
        .agg(**outer_agg_spec)
        .reset_index()
        .sort_values(['outer_fold', 'outer_role'])
    )
    if 'positive_rows' in outer_summary_df.columns:
        outer_summary_df['positive_rate_pct'] = (outer_summary_df['positive_rows'] / outer_summary_df['rows'] * 100).round(5)

    inner_source_df = nested_summary_source_df.loc[~nested_summary_source_df['split_role'].eq('outer_test')].copy()
    inner_agg_spec = {'rows': ('isic_id', 'nunique'), 'patients': ('patient_id', 'nunique')}
    if 'target' in inner_source_df.columns:
        inner_agg_spec['positive_rows'] = ('target', 'sum')
    inner_summary_df = (
        inner_source_df.groupby(['outer_fold', 'inner_fold', 'split_role'], dropna=False)
        .agg(**inner_agg_spec)
        .reset_index()
        .sort_values(['outer_fold', 'inner_fold', 'split_role'])
    )
    if 'positive_rows' in inner_summary_df.columns:
        inner_summary_df['positive_rate_pct'] = (inner_summary_df['positive_rows'] / inner_summary_df['rows'] * 100).round(5)

    outer_overlap_rows = []
    inner_overlap_rows = []
    for outer_fold, outer_frame in nested_split_df.groupby('outer_fold', dropna=False):
        outer_test_patients = set(outer_frame.loc[outer_frame['split_role'].eq('outer_test'), 'patient_id'].astype(str))
        cv_train_patients = set(outer_frame.loc[~outer_frame['split_role'].eq('outer_test'), 'patient_id'].astype(str))
        outer_overlap_rows.append(
            {
                'outer_fold': int(outer_fold),
                'cv_train_outer_test_patient_overlap': len(cv_train_patients & outer_test_patients),
            }
        )
        for inner_fold, inner_frame in outer_frame.groupby('inner_fold', dropna=False):
            inner_train_patients = set(inner_frame.loc[inner_frame['split_role'].eq('inner_train'), 'patient_id'].astype(str))
            inner_validation_patients = set(inner_frame.loc[inner_frame['split_role'].eq('inner_validation'), 'patient_id'].astype(str))
            inner_overlap_rows.append(
                {
                    'outer_fold': int(outer_fold),
                    'inner_fold': int(inner_fold),
                    'inner_train_inner_validation_patient_overlap': len(inner_train_patients & inner_validation_patients),
                    'inner_train_outer_test_patient_overlap': len(inner_train_patients & outer_test_patients),
                    'inner_validation_outer_test_patient_overlap': len(inner_validation_patients & outer_test_patients),
                }
            )
    outer_overlap_df = pd.DataFrame(outer_overlap_rows)
    inner_overlap_df = pd.DataFrame(inner_overlap_rows)

    display(Markdown('**표 5-a. Nested CV split schema audit**'))
    display(nested_schema_audit_df)
    display(Markdown('**표 5-b. outer cv_test_fold / cv_train summary**'))
    display(outer_summary_df)
    display(Markdown('**표 5-c. inner_train / inner_validation summary**'))
    display(inner_summary_df)
    display(Markdown('**표 5-d. outer patient overlap audit**'))
    display(outer_overlap_df)
    display(Markdown('**표 5-e. inner patient overlap audit**'))
    display(inner_overlap_df)

    if not nested_schema_audit_df['pass'].all():
        raise AssertionError('Nested split schema audit failed')
    if not (outer_overlap_df.drop(columns=['outer_fold']) == 0).all().all():
        raise AssertionError('Outer patient overlap audit failed')
    if not (inner_overlap_df.drop(columns=['outer_fold', 'inner_fold']) == 0).all().all():
        raise AssertionError('Inner patient overlap audit failed')


## 6. Summary evidence

현재 summary JSON이 존재하면 Nested CV protocol evidence를 읽는다. 이 값은 export CLI가 같은 nested split 계약을 생성했는지 확인하는 작은 evidence다.


In [ ]:
if summary is None:
    summary_overview_df = pd.DataFrame(
        [{'item': 'summary_json', 'value': 'missing_or_pending', 'status': 'Run export CLI after Nested CV migration'}]
    )
    leakage_controls_df = pd.DataFrame()
else:
    summary_overview_df = pd.DataFrame(
        [
            {'item': 'dataset_name', 'value': summary.get('dataset_name')},
            {'item': 'protocol_version', 'value': summary.get('protocol_version')},
            {'item': 'seed', 'value': summary.get('seed')},
            {'item': 'outer_folds', 'value': summary.get('outer_folds')},
            {'item': 'inner_folds', 'value': summary.get('inner_folds')},
            {'item': 'strict_input_column_count', 'value': summary.get('strict_input_contract', {}).get('strict_input_column_count')},
            {'item': 'inference_requires_iddx_full', 'value': summary.get('strict_input_contract', {}).get('inference_requires_iddx_full')},
            {'item': 'nested_split_output', 'value': summary.get('outputs', {}).get('nested_split_output')},
        ]
    )
    leakage_controls_df = pd.DataFrame(
        [{'control': key, 'value': value} for key, value in summary.get('leakage_controls', {}).items()]
    )

required_metrics_df = pd.DataFrame(
    [
        {'metric': 'pAUC@TPR>=0.80', 'required': True, 'threshold_source': 'not threshold-dependent'},
        {'metric': 'AUC', 'required': True, 'threshold_source': 'not threshold-dependent'},
        {'metric': 'F1', 'required': True, 'threshold_source': 'inner_validation only'},
        {'metric': 'precision', 'required': True, 'threshold_source': 'inner_validation only'},
        {'metric': 'recall', 'required': True, 'threshold_source': 'inner_validation only'},
        {'metric': 'balanced_accuracy', 'required': True, 'threshold_source': 'inner_validation only'},
        {'metric': 'Average Precision / PR-AUC', 'required': True, 'threshold_source': 'not threshold-dependent'},
    ]
)

implementation_status_df = pd.DataFrame(
    [
        {'item': 'strict input export', 'status': 'available' if strict_df is not None else 'PENDING_EXPORT_CLI'},
        {'item': 'nested split export', 'status': 'available' if nested_split_df is not None else 'PENDING_EXPORT_CLI'},
        {'item': 'tabular/image runner nested preflight', 'status': 'expected in runner summary'},
        {'item': 'multimodal runner nested preflight', 'status': 'scaffold; must keep same artifact'},
    ]
)

display(Markdown('**표 6-a. export summary overview**'))
display(summary_overview_df)
if not leakage_controls_df.empty:
    display(Markdown('**표 6-b. leakage controls recorded by current CLI summary**'))
    display(leakage_controls_df)
display(Markdown('**표 6-c. required metrics and threshold source**'))
display(required_metrics_df)
display(Markdown('**표 6-d. Nested CV implementation status**'))
display(implementation_status_df)


## 7. 최종 해석

A paper-facing Nested CV result is valid only if:

```text
patient-level split is disjoint at outer and inner levels
outer and inner splits are generated with the Triple Stratified objective
preprocessing, class weights, and samplers are fit on inner_train only
model choice, early stopping, threshold, and calibration use inner_validation only
outer_test / cv_test_fold is final evaluation only
iddx_full is excluded from ordinary inference input
required metrics are reported fold-wise
```

Nested split artifact가 없으면 아직 export를 재실행하지 않은 상태다. 이 경우 결과표나 논문 주장에는 “Nested CV protocol code-backed”라고 쓰면 안 된다.
